# End to end — grounded work-order classification with Stirrup

Retrieve a real work order through MCP and classify its failure-code description without writing to
the database. The notebook discovers an existing work-order number first, so placeholders and missing
records cannot produce hallucinated classifications.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")

REPO = find_repo()
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
TRACE_DIR = ARTIFACTS / "trajectories"
LOG_DIR = ARTIFACTS / "logs"
TRACE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("python:", sys.version.split()[0])


repo: /Users/chathurangishyalika/IBM/AssetOpsBench
python: 3.12.13


In [ ]:
# Load environment variables from .env file in the repository root
from dotenv import load_dotenv

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None
    
repo = find_repo()

if repo:
    load_dotenv(repo / ".env", override=False)

## 1. Create a read-only MCP client

`AOB_READONLY=1` removes work-order write tools from the server exposed to both this preflight and
Stirrup.


In [2]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MCP_ENV = os.environ.copy()
MCP_ENV["AOB_READONLY"] = "1"

async def call_wo(tool_name, **arguments):
    params = StdioServerParameters(
        command="uv",
        args=["run", "--directory", str(REPO), "wo-mcp-server"],
        cwd=str(REPO),
        env=MCP_ENV,
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
    text = "\n".join(getattr(item, "text", str(item)) for item in result.content)
    try:
        payload = json.loads(text)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Non-JSON response from {tool_name}: {text[:500]}") from exc
    if isinstance(payload, dict) and payload.get("error"):
        raise RuntimeError(f"{tool_name} failed: {payload['error']}")
    return payload


## 2. Discover a real work order

The preferred TST benchmark record is used only if it exists. Otherwise, the notebook searches all
loaded sites and selects the first real record together with its actual site ID. This keeps the demo
grounded even when the TST scenario dataset has not been loaded.


In [3]:
PREFERRED_SITE_ID = "TST"
PREFERRED_WONUM = "TST-WO00032"

listing = await call_wo(
    "list_workorders",
    page_size=0,
    page_num=1,
)
work_orders = [
    row for row in listing.get("work_orders", [])
    if row.get("wonum") and row.get("siteid")
]
if not work_orders:
    raise RuntimeError(
        "No work orders are loaded. Run: uv run python src/couchdb/init_data.py"
    )

preferred = next(
    (
        row for row in work_orders
        if row.get("siteid") == PREFERRED_SITE_ID
        and row.get("wonum") == PREFERRED_WONUM
    ),
    None,
)
selected = preferred or work_orders[0]
SITE_ID = str(selected["siteid"])
WORK_ORDER_NUMBER = str(selected["wonum"])

print("work orders found across all sites:", len(work_orders))
print("preferred TST work order present:", preferred is not None)
print("selected site:", SITE_ID)
print("selected work order:", WORK_ORDER_NUMBER)


work orders found across all sites: 3
preferred TST work order present: False
selected site: NORTH
selected work order: 1000050


In [4]:
selected_record = await call_wo(
    "get_workorder",
    site_id=SITE_ID,
    wonum=WORK_ORDER_NUMBER,
)
selected_record


{'work_order': {'wonum': '1000050',
  'description': 'Quarterly preventive maintenance - AHU 2',
  'description_longdescription': None,
  'siteid': 'NORTH',
  'orgid': None,
  'assetnum': 'AHU2',
  'location': 'NORTH-ROOF',
  'status': 'COMP',
  'status_date': None,
  'worktype': 'PM',
  'wopriority': 3,
  'reportdate': '2020-06-01T08:00:00+00:00',
  'reportedby': 'PLANNER1',
  'failurecode': None,
  'parent': None,
  'taskid': None,
  'lead': None,
  'jpnum': None,
  'schedstart': '2020-06-15T08:00:00+00:00',
  'schedfinish': '2020-06-15T17:00:00+00:00',
  'targstartdate': None,
  'targcompdate': None,
  'actstart': None,
  'actfinish': '2020-06-15T16:30:00+00:00',
  'estlabhrs': 8.0,
  'actlabhrs': 7.5,
  'estlabcost': None,
  'actlabcost': 562.5,
  'estmatcost': None,
  'actmatcost': 95.0,
  'estservcost': None,
  'actservcost': None,
  'esttoolcost': None,
  'acttoolcost': None,
  'estatapprtotalcost': None,
  'esttotalcost': None,
  'acttotalcost': 657.5,
  'wplabor': [{'laborcode

## 3. Configure Stirrup

The default uses a LiteLLM-proxy model with reliable native tool calling. Stirrup remains the agent
framework. Override `KDD_MODEL_ID` only with another model known to execute structured tool calls.
Credentials are checked without printing secret values.


In [5]:
assert shutil.which("uv"), "Install uv first."
MODEL_ID = os.getenv(
    "KDD_MODEL_ID",
    "litellm_proxy/aws/claude-opus-4-8",
)
if MODEL_ID.startswith("watsonx/"):
    required_credentials = ["WATSONX_APIKEY", "WATSONX_PROJECT_ID"]
elif MODEL_ID.startswith("tokenrouter/"):
    required_credentials = ["TOKENROUTER_API_KEY", "TOKENROUTER_BASE_URL"]
elif MODEL_ID.startswith("litellm_proxy/"):
    required_credentials = ["LITELLM_API_KEY", "LITELLM_BASE_URL"]
else:
    required_credentials = []
missing_credentials = [name for name in required_credentials if not os.getenv(name)]
print("agent framework: Stirrup")
print("model:", MODEL_ID)
print("credentials:", "ready" if not missing_credentials else "missing " + ", ".join(missing_credentials))
if MODEL_ID.startswith("watsonx/"):
    print("WARNING: the tested WatsonX model often serialized tool calls as text instead of executing them.")


agent framework: Stirrup
model: litellm_proxy/aws/claude-opus-4-8
credentials: ready


## 4. Build the grounded question

The prompt contains a real number discovered above. It forbids simulation and all writes. If retrieval
fails, the required answer is `NOT FOUND`, preventing an unsupported classification.


In [18]:
ALLOWED_DESCRIPTIONS = [
    "Breakdown",
    "Electrical",
    "Fail to function",
    "Leaking",
    "Low output",
    "Minor in-service problems",
    "Overheating",
    "Plugged / choked",
    "Structural deficiency",
    "Vibration",
]

QUESTION = f"""Call wo__get_workorder with exactly:
- site_id: "{SITE_ID}"
- wonum: "{WORK_ORDER_NUMBER}"

You must execute the tool. Never print, simulate, or assume a tool result.
Do not modify the site or work-order number.

If retrieval returns an error, output exactly:
NOT FOUND

Read the returned work-order description and failure-code field.

If an existing failure-code description is present, return it exactly as stored.
If the record contains a failure-code identifier instead of its description, use the
read-only wo__get_failure_codes tool to resolve its description.

If no failure code is recorded, select exactly one of these descriptions based only
on the retrieved work-order description:
{chr(10).join(ALLOWED_DESCRIPTIONS)}

Do not call generate_work_order, update_workorder, approve_workorder,
assign_technician, close_workorder, cancel_workorder, or any other write tool.

After retrieving the work order, your next response must contain exactly the selected
failure-code description, or NOT FOUND when retrieval failed.

Do not explain your selection.
Do not mention the work order or failure-code field.
Do not use quotes, Markdown, or additional text.
Call finish with reason set to exactly the same selected description.
"""

print(QUESTION)

Call wo__get_workorder with exactly:
- site_id: "NORTH"
- wonum: "1000050"

You must execute the tool. Never print, simulate, or assume a tool result.
Do not modify the site or work-order number.

If retrieval returns an error, output exactly:
NOT FOUND

Read the returned work-order description and failure-code field.

If an existing failure-code description is present, return it exactly as stored.
If the record contains a failure-code identifier instead of its description, use the
read-only wo__get_failure_codes tool to resolve its description.

If no failure code is recorded, select exactly one of these descriptions based only
on the retrieved work-order description:
Breakdown
Electrical
Fail to function
Leaking
Low output
Minor in-service problems
Overheating
Plugged / choked
Structural deficiency
Vibration

Do not call generate_work_order, update_workorder, approve_workorder,
assign_technician, close_workorder, cancel_workorder, or any other write tool.

After retrieving the work o

In [19]:
!uv run direct-llm-agent \
  --model-id litellm_proxy/aws/claude-opus-4-8 \
  "Reply with exactly: OK"


────────────────────────────────────────────────────────────
  Answer
────────────────────────────────────────────────────────────
OK



In [20]:
!uv run stirrup-agent \
  --no-code \
  --max-turns 1 \
  --model-id litellm_proxy/aws/claude-opus-4-8 \
  "Reply with exactly OK. Do not call any tools."

[08/05/26 09:16:07] StirrupAgentRunner: starting                                
                    (model=litellm_proxy/aws/claude-opus-4-8, code=False,       
                    backend=docker, workspace=None, preserve=False)             
─────────────────────── ▶ assetops (aws/claude-opus-4-8) ───────────────────────

Agent Task:
▰▱▱▱▱▱▱ Running assetops  │  0/1 steps  │  0 tool calls  │  0 input tokens  │  0
tokens
▰▱▱▱▱▱▱ Running assetops  │  0/1 steps  │  0 tool calls  │  0 input tokens  │  0
Reply with exactly OK. Do not call any tools.
▰▱▱▱▱▱▱ Running assetops  │  0/1 steps  │  0 tool calls  │  0 input tokens  │  0
tokens
▰▱▱▱▱▱▱ Running assetops  │  0/1 steps  │  0 tool calls  │  0 input tokens  │  0
Warnings
▰▱▱▱▱▱▱ Running assetops  │  0/1 steps  │  0 tool calls  │  0 input tokens  │  0
tokens
▰▱▱▱▱▱▱ Running assetops  │  0/1 steps  │  0 tool calls  │  0 input tokens  │  0
⚠ Missing default tool: LocalCodeExecToolProvider
▰▱▱▱▱▱▱ Running assetops  │  0/1 steps  │  0 tool c

## 5. Run Stirrup and persist the trajectory

Complete logs stay on disk. The notebook uses the persisted trajectory as the authoritative result,
so Rich CLI output cannot cause JSON or Jupyter IOPub failures.


In [21]:
RUN_AGENT = True
AGENT_TIMEOUT_SECONDS = int(os.getenv("KDD_AGENT_TIMEOUT_SECONDS", "240"))
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = f"kdd-stirrup-wo-{stamp}"
SCENARIO_ID = "kdd-workorder-failure-code-001"
trajectory_path = TRACE_DIR / f"{RUN_ID}.json"
stdout_path = LOG_DIR / f"{RUN_ID}.stdout.log"
stderr_path = LOG_DIR / f"{RUN_ID}.stderr.log"
RUN_COMPLETED = False

if missing_credentials:
    raise RuntimeError("Configure credentials and restart the kernel: " + ", ".join(missing_credentials))
if not RUN_AGENT:
    print("Agent skipped because RUN_AGENT=False.")
else:
    env = MCP_ENV.copy()
    env["AGENT_TRAJECTORY_DIR"] = str(TRACE_DIR)
    cmd = [
        "uv", "run", "--directory", str(REPO), "stirrup-agent",
        "--no-code", "--json", "--max-turns", "4",
        "--model-id", MODEL_ID,
        "--run-id", RUN_ID,
        "--scenario-id", SCENARIO_ID,
        QUESTION,
    ]
    try:
        completed = subprocess.run(
            cmd, env=env, text=True, capture_output=True,
            timeout=AGENT_TIMEOUT_SECONDS,
        )
        stdout_text = completed.stdout or ""
        stderr_text = completed.stderr or ""
        returncode = completed.returncode
    except subprocess.TimeoutExpired as exc:
        stdout_text = exc.stdout or ""
        stderr_text = exc.stderr or ""
        if isinstance(stdout_text, bytes):
            stdout_text = stdout_text.decode(errors="replace")
        if isinstance(stderr_text, bytes):
            stderr_text = stderr_text.decode(errors="replace")
        returncode = None
        print(f"Agent timed out after {AGENT_TIMEOUT_SECONDS}s; partial logs were saved.")

    stdout_path.write_text(stdout_text, encoding="utf-8")
    stderr_path.write_text(stderr_text, encoding="utf-8")

    if returncode not in (0, None):
        tail = "\n".join(stderr_text.splitlines()[-25:])
        print(f"Agent exited with {returncode}. Last stderr lines:\n{tail}")
    elif returncode == 0:
        RUN_COMPLETED = True

    if trajectory_path.exists():
        RUN_COMPLETED = True
        print("trajectory recovered:", trajectory_path)
    else:
        print("No trajectory was persisted. Try another configured tool-calling model.")
        print("stdout log:", stdout_path)
        print("stderr log:", stderr_path)


trajectory recovered: /Users/chathurangishyalika/IBM/AssetOpsBench/artifacts/kdd_tutorial/trajectories/kdd-stirrup-wo-20260805T131613Z.json


## 6. Audit grounding, safety, and output

A proposed tool call written in model text is not execution. Only entries inside `tool_calls` count.


In [22]:
if not trajectory_path.exists():
    trajectory = {}
    retrieval_ok = False
    write_calls = []
    answer_allowed = False
    answer = ""
    audit = {
        "selected_work_order": WORK_ORDER_NUMBER,
        "actual_tool_calls": [],
        "retrieval_grounded": False,
        "write_calls": [],
        "final_answer": "",
        "single_line_answer": False,
        "answer_shape_valid": False,
        "run_valid": False,
    }
else:
    trajectory = json.loads(trajectory_path.read_text(encoding="utf-8"))
    turns = trajectory.get("trajectory", {}).get("turns", [])
    calls = [call for turn in turns for call in turn.get("tool_calls", [])]
    call_names = [call.get("name") for call in calls]

    retrieval_calls = [call for call in calls if call.get("name") == "wo__get_workorder"]
    write_tools = {
        "wo__generate_work_order", "wo__update_workorder", "wo__approve_workorder",
        "wo__assign_technician", "wo__close_workorder", "wo__cancel_workorder",
    }
    write_calls = sorted(write_tools.intersection(call_names))

    retrieval_ok = False
    if retrieval_calls:
        last = retrieval_calls[-1]
        exact_input = last.get("input") == {"site_id": SITE_ID, "wonum": WORK_ORDER_NUMBER}
        try:
            retrieval_output = json.loads(last.get("output", "{}"))
        except json.JSONDecodeError:
            retrieval_output = {"error": "non-JSON tool output"}
        retrieval_ok = exact_input and not retrieval_output.get("error")

    answer = str(trajectory.get("answer", "")).strip()
    single_line = bool(answer) and "\n" not in answer
    record = selected_record.get("work_order", selected_record)
    existing_failure = record.get("failure_code") or record.get("failurecode")
    answer_allowed = single_line and (
        answer == "NOT FOUND"
        or bool(existing_failure)
        or answer in ALLOWED_DESCRIPTIONS
    )

    audit = {
        "selected_work_order": WORK_ORDER_NUMBER,
        "actual_tool_calls": call_names,
        "retrieval_grounded": retrieval_ok,
        "write_calls": write_calls,
        "final_answer": answer,
        "single_line_answer": single_line,
        "answer_shape_valid": answer_allowed,
        "run_valid": retrieval_ok and not write_calls and answer_allowed and answer != "NOT FOUND",
    }
    display(audit)


{'selected_work_order': '1000050',
 'actual_tool_calls': ['wo__get_workorder', 'finish'],
 'retrieval_grounded': True,
 'write_calls': [],
 'final_answer': 'Minor in-service problems',
 'single_line_answer': True,
 'answer_shape_valid': True,
 'run_valid': True}

In [23]:
if not audit["run_valid"]:
    print("RUN REJECTED: do not send this trajectory to evaluation or the leaderboard.")
    if not retrieval_ok:
        print("- The exact work order was not successfully retrieved through a real tool call.")
    if write_calls:
        print("- A forbidden write tool was called:", write_calls)
    if not answer_allowed:
        print("- The final answer is not a valid one-line failure-code description.")
else:
    print("RUN ACCEPTED:", answer)


RUN ACCEPTED: Minor in-service problems
